# Explore imagery around event dates

Run the inventory notebook or script first. Copy `config/event_windows.example.json` to `config/event_windows.json`, then set your UTC date windows and use the same AOI IDs as the inventory. This notebook reads local inventories and works offline.

In [ ]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd().resolve()
repo = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "scripts" / "csda_common.py").is_file()),
    None,
)
if repo is None:
    raise RuntimeError("Open the repository folder in VS Code or start Jupyter there.")
print(f"Kernel: {sys.executable}")

In [ ]:
import json

config_path = repo / "config" / "event_windows.json"
inventory_dir = repo / "outputs"

if not config_path.is_file():
    raise FileNotFoundError("Copy config/event_windows.example.json to config/event_windows.json and edit it.")
for area_id in json.loads(config_path.read_text(encoding="utf-8"))["areas"]:
    if not (inventory_dir / area_id / "inventory.csv").is_file():
        raise FileNotFoundError(f"Run the inventory for {area_id!r} first.")

subprocess.run(
    [sys.executable, str(repo / "scripts" / "event_windows.py"),
     "--config", str(config_path), "--inventory-root", str(inventory_dir)],
    cwd=repo,
    check=True,
);

## Review availability and candidate scenes

In [ ]:
from IPython.display import display
import pandas as pd

summary = pd.read_csv(inventory_dir / "event_window_summary.csv")
display(summary)

try:
    candidates = pd.read_csv(inventory_dir / "event_window_candidates.csv")
except pd.errors.EmptyDataError:
    candidates = pd.DataFrame()

if candidates.empty:
    print("No candidate records matched the configured areas and collections.")
else:
    display(candidates.head(25))